# Task 4 — The reconciliation identity (ladder + four wrong students)

Part A fills the ladder **from the files**. Two of the weight cells are blank
on purpose, and the three per-file figures do not add up to anything: the
rail file is in tonnes, and two of the three files have **text sitting in
the weight column**, which `SUM` silently skips.

Then the identity, which must come out at exactly zero:

```
staging  −  loaded  −  rejected  −  removed  =  0
```

Run all cells top to bottom from the `Task6/` folder.


In [1]:
import importlib.util
from pathlib import Path
import pandas as pd

NB_DIR = Path.cwd()
DATA = NB_DIR / "data"
print("notebook dir:", NB_DIR)
print("data dir exists:", DATA.exists())

# The pipeline script's helpers, loaded from the file sitting next to this
# notebook — the same strict rules as W6_su92-bsdsm-f23-018.py: plain-decimal
# numbers only, DD/MM/YYYY-or-ISO dates, positional extract (headers differ
# across the three files, so columns are read by position, not by name).
_spec = importlib.util.spec_from_file_location("w6pipe", NB_DIR / "W6_su92-bsdsm-f23-018.py")
w6 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(w6)
CONTRACT, PLAIN, STATUS_MAP = w6.CONTRACT, w6.PLAIN, w6.STATUS_MAP
norm_id, plain_number, parse_date = w6.norm_id, w6.plain_number, w6.parse_date
extract_csv, extract_rail = w6.extract_csv, w6.extract_rail
find_source_files, mode_key, bizkey = w6.find_source_files, w6.mode_key, w6.bizkey
print("pipeline helpers loaded from W6_su92-bsdsm-f23-018.py")

notebook dir: /home/hatch/workspace/data-warehouse/week6/Task6
data dir exists: True
pipeline helpers loaded from W6_su92-bsdsm-f23-018.py


In [2]:
from collections import Counter

marine_path, rail_path, road_path = find_source_files(DATA / "march")
staging, artifacts = [], []
for path, kind in ((marine_path, "csv"), (rail_path, "xlsx"), (road_path, "csv")):
    rows, arts = extract_csv(path, path.name) if kind == "csv" else extract_rail(path, path.name)
    staging.extend(rows)
    artifacts.extend(arts)
    print(f"  {path.name}: {len(rows)} rows")
for a in artifacts:
    print(f"  extract artifact (NOT a row): {a[0]} line {a[1]} — {a[2]}")
stg = pd.DataFrame(staging)[CONTRACT + ["SourceFile", "SourceLine", "SourceRow"]]
print("per-file rows:", dict(Counter(stg["SourceFile"])))
print("staging rows:", len(stg))
assert len(stg) == 1284, "staging must be 1,284 rows"


  marine_march.csv: 261 rows


  rail_march.xlsx: 315 rows
  road_march.csv: 708 rows
  extract artifact (NOT a row): road_march.csv line 710 — blank line
  extract artifact (NOT a row): road_march.csv line 711 — TOTAL footer
per-file rows: {'marine_march.csv': 261, 'rail_march.xlsx': 315, 'road_march.csv': 708}
staging rows: 1284


In [3]:
alias = {r["RawSpelling"].strip(): int(r["CityKey"]) for r in
         pd.read_csv(DATA / "alias_lookup.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_city = pd.read_csv(DATA / "dims" / "DimCity.csv", dtype=str, keep_default_na=False)
city_name = {int(r["CityKey"]): r["CityName"] for r in dim_city.to_dict("records")}
route = {(int(r["OriginCityKey"]), int(r["DestCityKey"])): int(r["RouteKey"]) for r in
         pd.read_csv(DATA / "dims" / "DimRoute.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_status = pd.read_csv(DATA / "dims" / "DimStatus.csv", dtype=str, keep_default_na=False)
status_names = {int(r["StatusKey"]): r["StatusName"] for r in dim_status.to_dict("records")}
print(f"alias spellings: {len(alias)}; cities: {len(city_name)}; routes: {len(route)}; statuses: {status_names}")

alias spellings: 23; cities: 15; routes: 21; statuses: {1: 'Delivered', 2: 'In Transit', 3: 'Delayed', -1: 'Unknown'}


In [4]:
# ---- REJECT: the four classes (disjoint for March; verified below) ----
wtext = stg["WeightKG"].map(lambda v: plain_number(v) is None and (v or "").strip() != "")
baddate = stg["ShipDate"].map(lambda v: parse_date(v) is None)
wneg = stg["WeightKG"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0)
blankid = stg["ShipmentID"].map(lambda v: norm_id(v) == "")
assert not (wtext & baddate).any() and not (wtext & wneg).any() and not (wtext & blankid).any()
assert not (baddate & wneg).any() and not (baddate & blankid).any() and not (wneg & blankid).any()
rejected = stg[wtext | baddate | wneg | blankid].copy()
loadable = stg[~(wtext | baddate | wneg | blankid)].copy()
print(f"rejected: {len(rejected)} (weight-text {int(wtext.sum())}, date-unparseable {int(baddate.sum())}, "
      f"weight-negative {int(wneg.sum())}, id-blank {int(blankid.sum())}); loadable: {len(loadable)}")

# ---- DEDUP: TRIM + UPPER ShipmentID, first occurrence kept ----
loadable["_n"] = loadable["ShipmentID"].map(norm_id)
seen_raw, byte_removed = set(), 0
for _, row in loadable.iterrows():
    k = tuple(row[c] for c in CONTRACT)
    if k in seen_raw:
        byte_removed += 1
    else:
        seen_raw.add(k)
deduped = loadable.drop_duplicates(subset=["_n"], keep="first").copy()
removed = len(loadable) - len(deduped)
print(f"duplicates removed: {removed} ({byte_removed} byte-identical + {removed - byte_removed} trailing-space-only)")
deduped = deduped.drop(columns=["_n"])

# ---- TRANSFORM + ROUTE ----
work = deduped.copy()
is_rail = work["SourceFile"].map(lambda f: "rail" in f.lower())
wnum = work["WeightKG"].map(plain_number)
wkg = wnum.copy()
wkg[is_rail] = wkg[is_rail] * 1000
work["WeightKG_num"] = wkg.map(lambda v: int(round(v)))
work["FreightCostPKR_num"] = work["FreightCostPKR"].map(lambda v: int(round(plain_number(v))))
work["ShipmentID"] = work.apply(
    lambda r: ("RL-" + norm_id(r["ShipmentID"])
               if "rail" in r["SourceFile"].lower() and not norm_id(r["ShipmentID"]).startswith("RL-")
               else norm_id(r["ShipmentID"])), axis=1)
work["ShipDate_parsed"] = work["ShipDate"].map(parse_date)
work["DateKey"] = work["ShipDate_parsed"].map(lambda d: int(d.strftime("%Y%m%d")))
work["StatusKey"] = work["StatusRaw"].map(lambda v: STATUS_MAP.get((v or "").strip().upper(), ("", -1))[1])
work["OriginCityKey"] = work["Origin"].map(lambda v: alias.get((v or "").strip(), -1))
work["DestCityKey"] = work["Destination"].map(lambda v: alias.get((v or "").strip(), -1))
work["RouteKey"] = [route.get((o, d), -1) for o, d in zip(work["OriginCityKey"], work["DestCityKey"])]
work["ModeKey"] = work["SourceFile"].map(mode_key)

# ---- CONFORM CUSTOMERS: merge registry duplicates, keep the losing spelling as an alias ----
cust = pd.read_csv(DATA / "customers_master.csv", dtype=str, keep_default_na=False)
cust["num"] = cust["CustomerID"].map(lambda v: int(v.split("-")[1]))
cust["biz"] = cust["CustomerName"].map(bizkey)
groups = {}
for _, r in cust.iterrows():
    groups.setdefault(r["biz"], []).append(r)
alias_map, dim_rows = {}, []
for biz, members in groups.items():
    members = sorted(members, key=lambda r: r["num"])
    surv = members[0]
    for m in members[1:]:
        alias_map[m["num"]] = surv["num"]
    dim_rows.append({"CustomerKey": int(surv["num"]), "CustomerName": surv["CustomerName"],
                     "City": surv["City"], "Sector": surv["Sector"],
                     "Aliases": "; ".join(m["CustomerName"] for m in members[1:])})
work["CustomerKey"] = work["CustomerID"].map(
    lambda v: alias_map.get(int(v.split("-")[1]), int(v.split("-")[1])))
dim_customer = pd.DataFrame(dim_rows).sort_values("CustomerKey").reset_index(drop=True)
dim_customer = pd.concat([dim_customer, pd.DataFrame(
    [{"CustomerKey": -1, "CustomerName": "Unknown", "City": "Unknown", "Sector": "Unknown", "Aliases": ""}])],
    ignore_index=True)
print(f"registry rows: {len(cust)}; businesses after conform: {len(dim_customer) - 1}; merged pairs: {len(alias_map)}")

# ---- FACT (in memory; the pipeline script writes this to CSV) ----
fact = pd.DataFrame({
    "ShipmentID": work["ShipmentID"],
    "DateKey": work["DateKey"].astype(int),
    "OriginCityKey": work["OriginCityKey"].astype(int),
    "DestCityKey": work["DestCityKey"].astype(int),
    "RouteKey": work["RouteKey"].astype(int),
    "ModeKey": work["ModeKey"].astype(int),
    "StatusKey": work["StatusKey"].astype(int),
    "CustomerKey": work["CustomerKey"].astype(int),
    "WeightKG": work["WeightKG_num"].astype(int),
    "FreightCostPKR": work["FreightCostPKR_num"].astype(int),
    "ShipmentCount": 1})
print(f"fact rows: {len(fact)}; total weight: {int(fact['WeightKG'].sum()):,} kg; "
      f"total cost: PKR {int(fact['FreightCostPKR'].sum()):,}")

rejected: 24 (weight-text 9, date-unparseable 6, weight-negative 5, id-blank 4); loadable: 1260
duplicates removed: 60 (48 byte-identical + 12 trailing-space-only)


registry rows: 46; businesses after conform: 40; merged pairs: 6
fact rows: 1200; total weight: 17,542,050 kg; total cost: PKR 147,037,000


In [5]:
# Per-file figures AS EXTRACTED: plain-decimal SUMs only (text skipped, exactly
# as a spreadsheet SUM would skip it). Rail weights are TONNES at this stage.
def file_rows_and_sums(fname):
    sub = stg[stg["SourceFile"] == fname]
    w = sum(plain_number(v) for v in sub["WeightKG"] if plain_number(v) is not None)
    c = sum(plain_number(v) for v in sub["FreightCostPKR"] if plain_number(v) is not None)
    return len(sub), w, c

mar_n, mar_w, mar_c = file_rows_and_sums("marine_march.csv")
rail_n, rail_t, rail_c = file_rows_and_sums("rail_march.xlsx")
road_n, road_w, road_c = file_rows_and_sums("road_march.csv")
ladder = pd.DataFrame([
    ("marine_march.csv, as extracted", mar_n, f"{mar_w:,.0f}", f"{mar_c:,.0f}"),
    ("rail_march.xlsx, as extracted (TONNES)", rail_n, f"{rail_t:,.2f} t", f"{rail_c:,.0f}"),
    ("road_march.csv, as extracted", road_n, f"{road_w:,.0f}", f"{road_c:,.0f}"),
    ("staging, all three appended", len(stg), "— (types + units not fixed yet)", "—"),
    ("rejected", len(rejected), "—", "—"),
    ("removed as duplicate", removed, "—", "—"),
    ("loaded into FactShipment_March", len(fact), f"{int(fact['WeightKG'].sum()):,}", f"{int(fact['FreightCostPKR'].sum()):,}"),
], columns=["Stage", "Rows", "SUM(WeightKG)", "SUM(FreightCostPKR)"])
print(ladder.to_string(index=False))
identity = len(stg) - len(fact) - len(rejected) - removed
print(f"\nidentity: {len(stg)} - {len(fact)} - {len(rejected)} - {removed} = {identity}")

                                 Stage  Rows                   SUM(WeightKG) SUM(FreightCostPKR)
        marine_march.csv, as extracted   261                       4,683,000          35,571,000
rail_march.xlsx, as extracted (TONNES)   315                      9,218.25 t          88,702,500
          road_march.csv, as extracted   708                       4,487,200          32,259,500
           staging, all three appended  1284 — (types + units not fixed yet)                   —
                              rejected    24                               —                   —
                  removed as duplicate    60                               —                   —
        loaded into FactShipment_March  1200                      17,542,050         147,037,000

identity: 1284 - 1200 - 24 - 60 = 0


In [6]:
# Part B — four students, four pairs. Every one of them ran a pipeline that
# completed without an error. For each: what they did, and which T7 check
# catches it.
loadable["_n"] = loadable["ShipmentID"].map(norm_id)
loadable["_wkg"] = loadable.apply(lambda r: plain_number(r["WeightKG"]) * (1000 if "rail" in r["SourceFile"].lower() else 1), axis=1)
first_norm, seen3, trail_w = set(), set(), 0.0
for _, row in loadable.iterrows():
    k = tuple(row[c] for c in CONTRACT)
    byte_dup = k in seen3
    seen3.add(k)
    norm_dup = row["_n"] in first_norm
    if not norm_dup:
        first_norm.add(row["_n"])
    if norm_dup and not byte_dup:
        trail_w += row["_wkg"]
fact_w = int(fact["WeightKG"].sum())
rail_kg = int(fact.loc[fact["ModeKey"] == 2, "WeightKG"].sum())
city1_w = int(fact.loc[fact["OriginCityKey"] == -1, "WeightKG"].sum())

# Student 1: 1,212 rows / 17,713,950 kg — dedup on whole-row identity or ID as
# arrived, removed only the 48 byte-identical copies, missed the 12 trailing-space copies.
s1_rows = len(loadable) - byte_removed
s1_w = fact_w + int(trail_w)
print(f"1. {s1_rows} rows / {s1_w:,} kg — dedup before trim: removed {byte_removed}, missed 12 trailing-space copies")
print(f"   gap +{int(trail_w):,} kg = exactly those 12 rows' weight; caught by T7 check 3 (60 removed) and check 4 (1,200 rows)")

# Student 2: 1,200 rows / 8,603,247.75 kg — appended rail without converting tonnes to kg.
s2_w = fact_w - rail_kg + rail_kg / 1000
print(f"2. 1200 rows / {s2_w:,.2f} kg — rail contributed {rail_kg / 1000:,.2f} instead of {rail_kg:,} kg (gap -{rail_kg - rail_kg / 1000:,.2f})")
print("   caught by T7 check 5 (SUM weight)")

# Student 3: 1,209 rows / 17,542,050 kg — rejected only 15 rows, loaded the 9 text-weight rows.
s3_rows = len(stg) - 15 - removed
tw_cost = sum(plain_number(v) for v in rejected[wtext.loc[rejected.index]]["FreightCostPKR"])
print(f"3. {s3_rows} rows / {fact_w:,} kg — 1,209 = 1,200 + 9; weight text/NULL so SUM skips them and the weight total looks exact")
print(f"   caught by T7 check 4 (row count) and check 2 (24 rejects); note: those 9 rows' stated costs total PKR {tw_cost:,.0f},")
print("   so a cost total over the full table would expose them too — the weight total alone is the trap")

# Student 4: 1,186 rows / 17,346,900 kg — deleted the 14 Unknown-city rows instead of routing them.
s4_rows = len(fact) - int((fact["OriginCityKey"] == -1).sum())
s4_w = fact_w - city1_w
print(f"4. {s4_rows} rows / {s4_w:,} kg — 1,200 - 14; weight -{city1_w:,} kg")
print("   caught by T7 check 7 (Unknown city = 14) and check 4")

1. 1212 rows / 17,713,950 kg — dedup before trim: removed 48, missed 12 trailing-space copies
   gap +171,900 kg = exactly those 12 rows' weight; caught by T7 check 3 (60 removed) and check 4 (1,200 rows)
2. 1200 rows / 8,603,247.75 kg — rail contributed 8,947.75 instead of 8,947,750 kg (gap -8,938,802.25)
   caught by T7 check 5 (SUM weight)
3. 1209 rows / 17,542,050 kg — 1,209 = 1,200 + 9; weight text/NULL so SUM skips them and the weight total looks exact
   caught by T7 check 4 (row count) and check 2 (24 rejects); note: those 9 rows' stated costs total PKR 1,081,500,
   so a cost total over the full table would expose them too — the weight total alone is the trap
4. 1186 rows / 17,346,900 kg — 1,200 - 14; weight -195,150 kg
   caught by T7 check 7 (Unknown city = 14) and check 4


In [7]:
assert (mar_n, mar_w, mar_c) == (261, 4683000.0, 35571000.0)
assert (rail_n, rail_t, rail_c) == (315, 9218.25, 88702500.0)
assert (road_n, road_w, road_c) == (708, 4487200.0, 32259500.0)
assert identity == 0, f"identity must be exactly 0, got {identity}"
assert (s1_rows, s1_w) == (1212, 17713950)
assert abs(s2_w - 8603247.75) < 1e-6
assert s3_rows == 1209 and tw_cost == 1081500.0
assert (s4_rows, s4_w) == (1186, 17346900)
print("SELF-CHECK PASS — ladder exact, identity 0, all four wrong pairs diagnosed")

# Analysis
# The ladder is the only unforgivable outcome this week done right: any
# other number than zero means rows left the pipeline and nothing recorded
# where they went. A wrong number you can find; a missing row you cannot.

SELF-CHECK PASS — ladder exact, identity 0, all four wrong pairs diagnosed
